In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded successfully')

/Users/shivamverma/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Libraries loaded successfully


In [2]:
base = os.path.join(
    os.path.expanduser('~'), 'Desktop',
    'Credit-Risk-Analytics-Portfolio',
    '02_data_cleaning_pipeline'
)

# Find raw file automatically
raw_folder = os.path.join(base, 'data', 'raw')
files      = [f for f in os.listdir(raw_folder) if f.endswith('.csv')]
raw_file   = os.path.join(raw_folder, files[0])

# Load only columns needed for dashboard
usecols = [
    'loan_amnt', 'funded_amnt', 'term', 'int_rate',
    'installment', 'grade', 'sub_grade', 'emp_length',
    'home_ownership', 'annual_inc', 'loan_status',
    'dti', 'addr_state', 'issue_d', 'purpose',
    'verification_status', 'revol_bal',
    'revol_util', 'total_acc', 'open_acc',
    'delinq_2yrs', 'earliest_cr_line', 'pub_rec'
]

print('Loading data...')
df = pd.read_csv(
    raw_file,
    usecols=lambda col: col in usecols
)

# Store original shape for cleaning log
original_shape = df.shape

print(f'Raw data loaded')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loading data...
Raw data loaded
Shape: 2,260,701 rows x 23 columns


In [3]:
# Initialise cleaning log
# Every cleaning step will append an entry here
cleaning_log = []

def log_step(step, description, before, after, notes=''):
    """
    Record every cleaning action with before and after metrics.
    This creates a full audit trail of all changes made.
    """
    cleaning_log.append({
        'Step'       : step,
        'Description': description,
        'Before'     : before,
        'After'      : after,
        'Notes'      : notes
    })
    print(f'  Step {step}: {description}')
    print(f'    Before: {before}  |  After: {after}')
    if notes:
        print(f'    Note  : {notes}')

print('Cleaning log initialised')
print(f'Starting shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

Cleaning log initialised
Starting shape: 2,260,701 rows x 23 columns


In [4]:
rows_before = len(df)

# Remove rows where loan_amnt is missing or not a number
# These are footer/header rows accidentally included
df = df[pd.to_numeric(df['loan_amnt'], errors='coerce').notna()]
df = df.reset_index(drop=True)

rows_after = len(df)
log_step(
    1, 'Remove non-loan rows',
    f'{rows_before:,} rows',
    f'{rows_after:,} rows',
    'Removed footer/header rows where loan_amnt was not numeric'
)

  Step 1: Remove non-loan rows
    Before: 2,260,701 rows  |  After: 2,260,668 rows
    Note  : Removed footer/header rows where loan_amnt was not numeric


In [5]:
# Fix 1 — int_rate: remove % sign and convert to float
if 'int_rate' in df.columns and df['int_rate'].dtype == 'object':
    before_sample = df['int_rate'].dropna().iloc[0]
    df['int_rate'] = pd.to_numeric(
        df['int_rate'].str.replace('%', '').str.strip(),
        errors='coerce'
    )
    log_step(2, 'Fix int_rate type',
             f'object, sample: {before_sample}',
             f'float, sample: {df["int_rate"].dropna().iloc[0]}',
             'Removed % sign, converted to float')

# Fix 2 — revol_util: remove % sign and convert to float
if 'revol_util' in df.columns and df['revol_util'].dtype == 'object':
    before_sample = df['revol_util'].dropna().iloc[0]
    df['revol_util'] = pd.to_numeric(
        df['revol_util'].str.replace('%', '').str.strip(),
        errors='coerce'
    )
    log_step(3, 'Fix revol_util type',
             f'object, sample: {before_sample}',
             f'float, sample: {df["revol_util"].dropna().iloc[0]}',
             'Removed % sign, converted to float')

# Fix 3 — term: extract number from 36 months or 60 months
if 'term' in df.columns and df['term'].dtype == 'object':
    before_sample = df['term'].dropna().iloc[0]
    df['term'] = pd.to_numeric(
        df['term'].str.replace('months', '').str.strip(),
        errors='coerce'
    )
    log_step(4, 'Fix term type',
             f'object, sample: {before_sample}',
             f'float, sample: {df["term"].dropna().iloc[0]}',
             'Extracted number from text, dropped months label')

# Fix 4 — emp_length: extract number from 5 years, 10+ years etc.
if 'emp_length' in df.columns and df['emp_length'].dtype == 'object':
    before_sample = df['emp_length'].dropna().iloc[0]
    df['emp_length'] = df['emp_length'].str.replace('years', '')
    df['emp_length'] = df['emp_length'].str.replace('year',  '')
    df['emp_length'] = df['emp_length'].str.replace('+',     '')
    df['emp_length'] = df['emp_length'].str.replace('< 1',   '0')
    df['emp_length'] = pd.to_numeric(
        df['emp_length'].str.strip(),
        errors='coerce'
    )
    log_step(5, 'Fix emp_length type',
             f'object, sample: {before_sample}',
             f'float, sample: {df["emp_length"].dropna().iloc[0]}',
             'Extracted number, replaced 10+ with 10, < 1 year with 0')

print('\nType conversion complete')
print(df[['int_rate','revol_util','term','emp_length']].dtypes)

  Step 4: Fix term type
    Before: object, sample:  36 months  |  After: float, sample: 36
    Note  : Extracted number from text, dropped months label
  Step 5: Fix emp_length type
    Before: object, sample: 10+ years  |  After: float, sample: 10.0
    Note  : Extracted number, replaced 10+ with 10, < 1 year with 0

Type conversion complete
int_rate      float64
revol_util    float64
term            int64
emp_length    float64
dtype: object


In [6]:
# Standardise all text columns
# Strip whitespace and convert to consistent title case

text_cols = df.select_dtypes(include='object').columns.tolist()

for col in text_cols:
    df[col] = df[col].str.strip()

# Specific standardisations
# home_ownership - uppercase for consistency
if 'home_ownership' in df.columns:
    before = df['home_ownership'].value_counts().head(3).to_dict()
    df['home_ownership'] = df['home_ownership'].str.upper()
    after  = df['home_ownership'].value_counts().head(3).to_dict()
    log_step(6, 'Standardise home_ownership',
             str(before), str(after), 'Converted to uppercase')

# grade - uppercase
if 'grade' in df.columns:
    df['grade'] = df['grade'].str.upper().str.strip()

# purpose - lowercase with underscores replaced by spaces
if 'purpose' in df.columns:
    df['purpose'] = df['purpose'].str.lower().str.replace('_', ' ').str.title()

# verification_status - title case
if 'verification_status' in df.columns:
    df['verification_status'] = df['verification_status'].str.strip()

log_step(7, 'Standardise text columns',
         f'{len(text_cols)} text columns',
         'All stripped of whitespace',
         'grade=upper, purpose=title, home_ownership=upper')

print('Text standardisation complete')

  Step 6: Standardise home_ownership
    Before: {'MORTGAGE': 1111450, 'RENT': 894929, 'OWN': 253057}  |  After: {'MORTGAGE': 1111450, 'RENT': 894929, 'OWN': 253057}
    Note  : Converted to uppercase
  Step 7: Standardise text columns
    Before: 9 text columns  |  After: All stripped of whitespace
    Note  : grade=upper, purpose=title, home_ownership=upper
Text standardisation complete


In [7]:
# Parse issue_d - loan issue date
if 'issue_d' in df.columns:
    before_sample = df['issue_d'].dropna().iloc[0]

    df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')

    # Extract year and month as separate columns for dashboard filters
    df['issue_year']  = df['issue_d'].dt.year
    df['issue_month'] = df['issue_d'].dt.month
    df['issue_month_name'] = df['issue_d'].dt.strftime('%b')
    df['issue_quarter'] = df['issue_d'].dt.quarter.apply(lambda x: f'Q{x}')

    log_step(8, 'Parse issue_d',
             f'object: {before_sample}',
             f'datetime + year/month/quarter extracted',
             'Added issue_year, issue_month, issue_month_name, issue_quarter')

# Parse earliest_cr_line
if 'earliest_cr_line' in df.columns:
    df['earliest_cr_line'] = pd.to_datetime(
        df['earliest_cr_line'], format='%b-%Y', errors='coerce'
    )
    # Credit history length in years
    df['credit_history_years'] = (
        (df['issue_d'] - df['earliest_cr_line']).dt.days / 365
    ).round(1)

    log_step(9, 'Parse earliest_cr_line',
             'object date string',
             'datetime + credit_history_years derived',
             'Credit history = issue date minus earliest credit line')

print('Date parsing complete')
print(df[['issue_d','issue_year','issue_month','issue_quarter']].head(3))

  Step 8: Parse issue_d
    Before: object: Dec-2015  |  After: datetime + year/month/quarter extracted
    Note  : Added issue_year, issue_month, issue_month_name, issue_quarter
  Step 9: Parse earliest_cr_line
    Before: object date string  |  After: datetime + credit_history_years derived
    Note  : Credit history = issue date minus earliest credit line
Date parsing complete
     issue_d  issue_year  issue_month issue_quarter
0 2015-12-01        2015           12            Q4
1 2015-12-01        2015           12            Q4
2 2015-12-01        2015           12            Q4


In [8]:
rows_before = len(df)

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

rows_after = len(df)
removed    = rows_before - rows_after

log_step(10, 'Remove duplicate rows',
         f'{rows_before:,} rows',
         f'{rows_after:,} rows',
         f'Removed {removed:,} exact duplicate rows')

print(f'Duplicates removed: {removed:,}')

  Step 10: Remove duplicate rows
    Before: 2,260,668 rows  |  After: 2,260,668 rows
    Note  : Removed 0 exact duplicate rows
Duplicates removed: 0


In [9]:
# Missing value treatment strategy
# Numerical  -> median imputation
# Categorical -> fill with UNKNOWN or most frequent

print('Missing values before treatment:')
missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0]
print(missing_before)

# Numerical columns - median imputation
num_cols_with_missing = [
    'emp_length', 'dti', 'revol_util', 'annual_inc',
    'delinq_2yrs', 'open_acc', 'pub_rec', 'total_acc',
    'revol_bal', 'credit_history_years'
]

for col in num_cols_with_missing:
    if col in df.columns and df[col].isnull().sum() > 0:
        n_missing  = df[col].isnull().sum()
        median_val = df[col].median()
        df[col]    = df[col].fillna(median_val)
        log_step(
            f'11a', f'Impute {col}',
            f'{n_missing:,} missing',
            f'Filled with median = {median_val:.2f}',
            'Median robust to outliers'
        )

# Categorical columns - fill with UNKNOWN
cat_cols_with_missing = [
    'verification_status', 'purpose',
    'home_ownership', 'addr_state'
]

for col in cat_cols_with_missing:
    if col in df.columns and df[col].isnull().sum() > 0:
        n_missing = df[col].isnull().sum()
        df[col]   = df[col].fillna('Unknown')
        log_step(
            f'11b', f'Impute {col}',
            f'{n_missing:,} missing',
            'Filled with Unknown',
            'Preserves missingness as its own category'
        )

print('\nMissing values after treatment:')
missing_after = df.isnull().sum()
missing_after = missing_after[missing_after > 0]
if len(missing_after) == 0:
    print('  No missing values remaining')
else:
    print(missing_after)

Missing values before treatment:
emp_length              146907
annual_inc                   4
dti                       1711
delinq_2yrs                 29
earliest_cr_line            29
open_acc                    29
pub_rec                     29
revol_util                1802
total_acc                   29
credit_history_years        29
dtype: int64
  Step 11a: Impute emp_length
    Before: 146,907 missing  |  After: Filled with median = 6.00
    Note  : Median robust to outliers
  Step 11a: Impute dti
    Before: 1,711 missing  |  After: Filled with median = 17.84
    Note  : Median robust to outliers
  Step 11a: Impute revol_util
    Before: 1,802 missing  |  After: Filled with median = 50.30
    Note  : Median robust to outliers
  Step 11a: Impute annual_inc
    Before: 4 missing  |  After: Filled with median = 65000.00
    Note  : Median robust to outliers
  Step 11a: Impute delinq_2yrs
    Before: 29 missing  |  After: Filled with median = 0.00
    Note  : Median robust to out

In [10]:
# Winsorize key financial variables
winsorize_cols = ['annual_inc', 'dti', 'revol_bal', 'revol_util', 'installment']
winsorize_cols = [c for c in winsorize_cols if c in df.columns]

print('Outlier Treatment (Winsorization at 1st/99th percentile)')
print('='*60)

for col in winsorize_cols:
    p01 = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)

    n_lower  = (df[col] < p01).sum()
    n_upper  = (df[col] > p99).sum()

    df[col] = df[col].clip(lower=p01, upper=p99)

    print(f'  {col:<20} Cap: [{p01:.1f}, {p99:.1f}]  '
          f'Lower capped: {n_lower:,}  Upper capped: {n_upper:,}')

    log_step(12, f'Winsorize {col}',
             f'Uncapped range',
             f'Capped at [{p01:.1f}, {p99:.1f}]',
             f'Lower: {n_lower:,}, Upper: {n_upper:,} values capped')

print('\nOutlier treatment complete')

Outlier Treatment (Winsorization at 1st/99th percentile)
  annual_inc           Cap: [16800.0, 270000.0]  Lower capped: 22,545  Upper capped: 22,543
  Step 12: Winsorize annual_inc
    Before: Uncapped range  |  After: Capped at [16800.0, 270000.0]
    Note  : Lower: 22,545, Upper: 22,543 values capped
  dti                  Cap: [1.7, 42.7]  Lower capped: 22,538  Upper capped: 22,603
  Step 12: Winsorize dti
    Before: Uncapped range  |  After: Capped at [1.7, 42.7]
    Note  : Lower: 22,538, Upper: 22,603 values capped
  revol_bal            Cap: [126.0, 97930.0]  Lower capped: 22,591  Upper capped: 22,607
  Step 12: Winsorize revol_bal
    Before: Uncapped range  |  After: Capped at [126.0, 97930.0]
    Note  : Lower: 22,591, Upper: 22,607 values capped
  revol_util           Cap: [0.9, 98.1]  Lower capped: 22,530  Upper capped: 22,397
  Step 12: Winsorize revol_util
    Before: Uncapped range  |  After: Capped at [0.9, 98.1]
    Note  : Lower: 22,530, Upper: 22,397 values capped
 

In [11]:
# 1 — Simplified loan status (for traffic light reporting)
status_mapping = {
    'Fully Paid'                                         : 'Good',
    'Current'                                            : 'Current',
    'Charged Off'                                        : 'Bad',
    'Late (31-120 days)'                                 : 'Bad',
    'Issued'                                             : 'Current',
    'In Grace Period'                                    : 'Watch',
    'Late (16-30 days)'                                  : 'Watch',
    'Does not meet the credit policy. Status:Fully Paid' : 'Good',
    'Does not meet the credit policy. Status:Charged Off': 'Bad',
    'Default'                                            : 'Bad'
}
df['loan_status_simple'] = df['loan_status'].map(status_mapping).fillna('Other')

# 2 — Risk tier from grade (for dashboard grouping)
risk_mapping = {
    'A': 'Prime',
    'B': 'Near Prime',
    'C': 'Subprime',
    'D': 'Subprime',
    'E': 'Deep Subprime',
    'F': 'Deep Subprime',
    'G': 'Deep Subprime'
}
df['risk_tier'] = df['grade'].map(risk_mapping).fillna('Unknown')

# 3 — Loan size band (for volume analysis)
df['loan_size_band'] = pd.cut(
    df['loan_amnt'],
    bins=[0, 5000, 10000, 20000, 35000, 999999],
    labels=['<5K', '5K-10K', '10K-20K', '20K-35K', '>35K']
)

# 4 — Interest rate band
df['int_rate_band'] = pd.cut(
    df['int_rate'],
    bins=[0, 8, 12, 16, 20, 100],
    labels=['<8%', '8-12%', '12-16%', '16-20%', '>20%']
)

# 5 — DTI band (debt burden)
df['dti_band'] = pd.cut(
    df['dti'],
    bins=[-1, 10, 20, 30, 40, 999],
    labels=['<10', '10-20', '20-30', '30-40', '>40']
)

# 6 — Annual income band
df['income_band'] = pd.cut(
    df['annual_inc'],
    bins=[0, 40000, 70000, 100000, 150000, 9999999],
    labels=['<40K', '40-70K', '70-100K', '100-150K', '>150K']
)

# 7 — Is bad loan flag (binary for calculations)
df['is_bad'] = (df['loan_status_simple'] == 'Bad').astype(int)

# 8 — Monthly payment to income ratio (affordability metric)
df['payment_to_income'] = (
    (df['installment'] * 12) / (df['annual_inc'] + 1) * 100
).round(2)

log_step(13, 'Create derived columns',
         'Original columns only',
         '8 new columns added',
         'loan_status_simple, risk_tier, loan/int/dti/income bands, is_bad, payment_to_income')

print('Derived columns created')
print(f'New columns: loan_status_simple, risk_tier, loan_size_band, '
      f'int_rate_band, dti_band, income_band, is_bad, payment_to_income')

  Step 13: Create derived columns
    Before: Original columns only  |  After: 8 new columns added
    Note  : loan_status_simple, risk_tier, loan/int/dti/income bands, is_bad, payment_to_income
Derived columns created
New columns: loan_status_simple, risk_tier, loan_size_band, int_rate_band, dti_band, income_band, is_bad, payment_to_income


In [12]:
print('FINAL VALIDATION CHECKS')
print('='*55)

checks = []

# Check 1 — No missing values in critical columns
critical_cols = ['loan_amnt', 'int_rate', 'grade', 'loan_status', 'annual_inc']
critical_cols = [c for c in critical_cols if c in df.columns]
for col in critical_cols:
    missing = df[col].isnull().sum()
    status  = 'PASS' if missing == 0 else 'FAIL'
    checks.append({'Check': f'No nulls in {col}', 'Result': missing, 'Status': status})

# Check 2 — loan_amnt is always positive
neg_loans = (df['loan_amnt'] <= 0).sum()
checks.append({'Check': 'loan_amnt > 0', 'Result': neg_loans, 'Status': 'PASS' if neg_loans == 0 else 'FAIL'})

# Check 3 — int_rate is between 0 and 100
bad_rate = ((df['int_rate'] < 0) | (df['int_rate'] > 100)).sum()
checks.append({'Check': 'int_rate in [0,100]', 'Result': bad_rate, 'Status': 'PASS' if bad_rate == 0 else 'FAIL'})

# Check 4 — grade is only A-G
valid_grades  = set(['A','B','C','D','E','F','G'])
invalid_grade = (~df['grade'].isin(valid_grades)).sum()
checks.append({'Check': 'grade in A-G', 'Result': invalid_grade, 'Status': 'PASS' if invalid_grade == 0 else 'FAIL'})

# Check 5 — issue_year is reasonable
if 'issue_year' in df.columns:
    bad_year = ((df['issue_year'] < 2007) | (df['issue_year'] > 2025)).sum()
    checks.append({'Check': 'issue_year in [2007,2025]', 'Result': bad_year, 'Status': 'PASS' if bad_year == 0 else 'FAIL'})

# Check 6 — no duplicate rows
dupes = df.duplicated().sum()
checks.append({'Check': 'No duplicates', 'Result': dupes, 'Status': 'PASS' if dupes == 0 else 'FAIL'})

checks_df = pd.DataFrame(checks)
print(checks_df.to_string(index=False))

passes = (checks_df['Status'] == 'PASS').sum()
print(f'\nValidation: {passes}/{len(checks_df)} checks passed')

FINAL VALIDATION CHECKS
                    Check  Result Status
    No nulls in loan_amnt       0   PASS
     No nulls in int_rate       0   PASS
        No nulls in grade       0   PASS
  No nulls in loan_status       0   PASS
   No nulls in annual_inc       0   PASS
            loan_amnt > 0       0   PASS
      int_rate in [0,100]       0   PASS
             grade in A-G       0   PASS
issue_year in [2007,2025]       0   PASS
            No duplicates       0   PASS

Validation: 10/10 checks passed


In [ ]:
# Create processed folder if it does not exist
processed_folder = os.path.join(base, 'data', 'processed')
os.makedirs(processed_folder, exist_ok=True)

# Save full clean dataset
output_path = os.path.join(processed_folder, 'loans_clean.csv')
df.to_csv(output_path, index=False)

print(f'Clean dataset saved')
print(f'Path  : {output_path}')
print(f'Shape : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'\nColumns in clean dataset:')
for col in df.columns:
    print(f'  {col:<30} {str(df[col].dtype)}')

In [ ]:
# Save cleaning log to CSV
cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(
    os.path.join(base, 'outputs', 'cleaning_log.csv'), index=False
)

print('CLEANING LOG')
print('='*75)
print(cleaning_log_df[['Step','Description','Before','After']].to_string(index=False))
print(f'\nCleaning log saved')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Chart 1 — Loan status distribution
status_counts = df['loan_status_simple'].value_counts()
colors_status = {'Good':'#2ecc71','Bad':'#e74c3c',
                 'Current':'#3498db','Watch':'#f39c12','Other':'#95a5a6'}
bar_colors = [colors_status.get(s,'#95a5a6') for s in status_counts.index]
axes[0,0].bar(status_counts.index, status_counts.values,
              color=bar_colors, edgecolor='white')
axes[0,0].set_title('Loan Status Distribution', fontweight='bold')
axes[0,0].set_ylabel('Count')
for i, v in enumerate(status_counts.values):
    axes[0,0].text(i, v + 1000, f'{v/1000:.0f}K', ha='center', fontsize=9)

# Chart 2 — Grade distribution
grade_counts = df['grade'].value_counts().sort_index()
axes[0,1].bar(grade_counts.index, grade_counts.values,
              color='#3498db', edgecolor='white')
axes[0,1].set_title('Loan Grade Distribution', fontweight='bold')
axes[0,1].set_ylabel('Count')

# Chart 3 — Loans issued per year
if 'issue_year' in df.columns:
    year_counts = df['issue_year'].value_counts().sort_index()
    axes[1,0].plot(year_counts.index, year_counts.values,
                   'o-', color='#2ecc71', linewidth=2.5, markersize=7)
    axes[1,0].fill_between(year_counts.index, year_counts.values,
                           alpha=0.2, color='#2ecc71')
    axes[1,0].set_title('Loans Issued per Year', fontweight='bold')
    axes[1,0].set_ylabel('Count')
    axes[1,0].set_xlabel('Year')

# Chart 4 — Bad rate by grade
bad_by_grade = df.groupby('grade')['is_bad'].mean() * 100
bad_by_grade = bad_by_grade.sort_index()
axes[1,1].bar(bad_by_grade.index, bad_by_grade.values,
              color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(bad_by_grade))),
              edgecolor='white')
axes[1,1].set_title('Default Rate by Grade', fontweight='bold')
axes[1,1].set_ylabel('Default Rate %')
for i, v in enumerate(bad_by_grade.values):
    axes[1,1].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=9)

plt.suptitle('Clean Dataset Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(base, 'outputs', '05_clean_data_overview.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved')

In [ ]:
print('='*60)
print('  DATA CLEANING COMPLETE')
print('='*60)
print(f"""
INPUT
  Raw rows     : {original_shape[0]:,}
  Raw columns  : {original_shape[1]}

OUTPUT
  Clean rows   : {df.shape[0]:,}
  Clean columns: {df.shape[1]}

CLEANING STEPS APPLIED
  1.  Removed non-loan footer rows
  2.  Fixed int_rate  - removed % sign, converted to float
  3.  Fixed revol_util - removed % sign, converted to float
  4.  Fixed term       - extracted number from text
  5.  Fixed emp_length - extracted number from text
  6.  Standardised text columns - strip, uppercase, title case
  7.  Parsed issue_d  - extracted year, month, quarter
  8.  Parsed earliest_cr_line - derived credit_history_years
  9.  Removed duplicate rows
  10. Imputed numerical missing values with median
  11. Imputed categorical missing values with Unknown
  12. Winsorized outliers at 1st/99th percentile
  13. Created 8 derived columns for dashboard

DERIVED COLUMNS ADDED
  loan_status_simple  - Good/Bad/Current/Watch
  risk_tier           - Prime/Near Prime/Subprime/Deep Subprime
  loan_size_band      - <5K / 5K-10K / 10K-20K etc
  int_rate_band       - <8% / 8-12% / 12-16% etc
  dti_band            - <10 / 10-20 / 20-30 etc
  income_band         - <40K / 40-70K etc
  is_bad              - 1 if Bad, 0 otherwise
  payment_to_income   - monthly payment as % of annual income

OUTPUT FILES
  data/processed/loans_clean.csv
  outputs/cleaning_log.csv
  outputs/05_clean_data_overview.png
""")
print('NEXT: 03_feature_engineering.ipynb')
print('='*60)